In [3]:
!pip install fastmcp


[notice] A new release of pip is available: 23.2.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
!pip install pydantic_ai

  Obtaining dependency information for pydantic_ai from https://files.pythonhosted.org/packages/b8/c6/b85d5d8da15c616374a5bf566aaa5ba13142b0024fddfe97ac288641792b/pydantic_ai-1.104.0-py3-none-any.whl.metadata
  Obtaining dependency information for pydantic-ai-slim[ag-ui,anthropic,bedrock,cli,cohere,evals,fastmcp,google,groq,huggingface,logfire,mcp,mistral,openai,retries,spec,temporal,ui,vertexai,xai]==1.104.0 from https://files.pythonhosted.org/packages/6a/1d/c03cecf9c48040f750c6e5b4f027fb4935cc41d9b76f1217af7731f4dc2b/pydantic_ai_slim-1.104.0-py3-none-any.whl.metadata
  Obtaining dependency information for genai-prices>=0.0.56 from https://files.pythonhosted.org/packages/81/35/ce64112dcc6f406b3e290dcf57a97acfa2b7d3d0391979219cb9d4a9db6d/genai_prices-0.0.62-py3-none-any.whl.metadata
  Using cached genai_prices-0.0.62-py3-none-any.whl.metadata (7.1 kB)
  Obtaining dependency information for griffelib>=2.0 from https://files.pythonhosted.org/packages/11/8c/c9138d881c79aa0ea9ed83cbd58d5ca

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
deepeval 2.8.9 requires anthropic<0.50.0,>=0.49.0, but you have anthropic 0.105.2 which is incompatible.
deepeval 2.8.9 requires google-genai<2.0.0,>=1.9.0, but you have google-genai 2.7.0 which is incompatible.
fastembed 0.4.0 requires huggingface-hub<1.0,>=0.20, but you have huggingface-hub 1.17.0 which is incompatible.
google-auth-oauthlib 1.2.3 requires google-auth<2.42.0,>=2.15.0, but you have google-auth 2.53.0 which is incompatible.
langchain-core 0.3.51 requires packaging<25,>=23.2, but you have packaging 25.0 which is incompatible.
langchain-google-genai 2.1.2 requires google-ai-generativelanguage<0.7.0,>=0.6.16, but you have google-ai-generativelanguage 0.6.15 which is incompatible.
langchain-openai 0.3.12 requires openai<2.0.0,>=1.68.2, but you have openai 2.38.0 which is incompatible.
mlflow 3.12.0 req

In [1]:
import os
import json
import sys

# 1. APPLY WINDOWS JUPYTER SUBPROCESS PATCH
# This restores the OS-level stderr file descriptor, preventing the 'fileno' error.
sys.stderr = sys.__stderr__
import random
import asyncio
import textwrap
from dataclasses import dataclass
from datetime import datetime
from typing import Optional, Any
import pandas as pd
import mlflow
import mlflow.entities
import mlflow.data
from openai import OpenAI
from pydantic import BaseModel, Field, ValidationError
import random
import functools
from pydantic_ai import Agent, ModelRetry
from pydantic_ai.agent import RunContext
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider
from pydantic_ai.profiles import ModelProfile
from pydantic_ai.settings import ModelSettings
from functools import partial
import matplotlib.pyplot as plt
import seaborn as sns

from mlflow.genai.scorers import (
    Correctness,
    Guidelines,
    ExpectationsGuidelines,
    scorer,
    ScorerSamplingConfig,
)
from mlflow.metrics.genai import (
    EvaluationExample,
    faithfulness,
    answer_correctness,
)
from mlflow.data.pandas_dataset import PandasDataset
from mlflow.genai.datasets import create_dataset, get_dataset
from mlflow.genai import make_judge

from mlflow.entities import Feedback
from mlflow import MlflowClient

from pydantic_ai import Agent
from pydantic_ai.mcp import MCPToolset
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider
from pydantic_ai.profiles import ModelProfile
from pydantic_ai.settings import ModelSettings

In [2]:
import litellm
litellm.set_verbose = True  

In [3]:

os.environ["OPENAI_API_BASE"] = "https://apphubai.wolke.uni-greifswald.de/v1"  #"https://apphubai.wolke.uni-greifswald.de/v1"#"http://models.system-service-ai/v1" 
os.environ["OPENAI_API_KEY"] = "RYpNq6AnGTbyaWX8ijFzl5tAdjNqcxWo"   #"RYpNq6AnGTbyaWX8ijFzl5tAdjNqcxWo" # "not-needed"   
os.environ["LITELLM_EXTRA_BODY"] = '{"chat_template_kwargs": {"enable_thinking": false}}'

model_id = "gemma3:27b"#"gemma3:27b" #"Qwen/Qwen3-VL-30B-A3B-Instruct-FP8" #"RedHatAI/Qwen3-32B-quantized.w4a16"
judge_uri = f"openai:/{model_id}"



In [ ]:

mlflow.set_tracking_uri("http://localhost:8050") 
exp_name = "Job_CV_Evaluation"

dataset_job_description = "eval_datset_job_description"
dataset_CV_description = "eval_datset_CV_description"
dataset_Matcher_description = "eval_datset_Matcher_description"

mlflow.set_experiment(exp_name)

experiment = mlflow.get_experiment_by_name(exp_name)
exp_id = experiment.experiment_id
#mlflow.set_tracking_uri("sqlite:///mlflow.db") 




In [7]:

def load_jobbank_jsonl(path):
    """
    Loads a JSONL file with fields:
    - url
    - title
    - full_page_text
    into a pandas DataFrame.
    """
    df = pd.read_json(path, lines=True)
    return df




In [10]:
df = load_jobbank_jsonl("data_collection/jobbank_jobs.jsonl")
df

,url,title,full_page_text
0,https://www.jobbank.gc.ca/jobsearch/jobposting...,software engineer,"software engineer - Mississauga, ON - Job post..."
1,https://www.jobbank.gc.ca/jobsearch/jobposting...,software engineer,"software engineer - Toronto, ON - Job posting ..."
2,https://www.jobbank.gc.ca/jobsearch/jobposting...,software engineer,"software engineer - Hamilton, ON - Job posting..."
3,https://www.jobbank.gc.ca/jobsearch/jobposting...,lead software engineer,"lead software engineer - Richmond Hill, ON - J..."
4,https://www.jobbank.gc.ca/jobsearch/jobposting...,software engineer,"software engineer - Vancouver, BC - Job postin..."
...,...,...,...
226,https://www.jobbank.gc.ca/jobsearch/jobposting...,graphic designer,"graphic designer - Calgary, AB - Job posting -..."
227,https://www.jobbank.gc.ca/jobsearch/jobposting...,graphic designer,"graphic designer - Coquitlam, BC - Job posting..."
228,https://www.jobbank.gc.ca/jobsearch/jobposting...,graphic designer,"graphic designer - Québec, QC - Job posting - ..."
229,https://www.jobbank.gc.ca/jobsearch/jobposting...,graphic designer,"graphic designer - Burnaby, BC - Job posting -..."


In [13]:
print(df["full_page_text"][1])

software engineer - Toronto, ON - Job posting - Job Bank
Skip to job search
Skip to main content
Skip to "About this Web application"
Language selection
Français
fr
Government of Canada /
Gouvernement du Canada
Search
Search website
Search
Job Bank
Job Bank
Account menu
Sign in
Job seekers
Employers
Menu and search
Menu
Menu
Account menu
Sign in
Job seekers
Employers
Main navigation menu
Job search
Training and careers
Labour market information
Hiring
Help
About
You are here:
Job Bank
Dashboard
Search
Job Search
Keywords:
Location:
All of Canada
Current location
Search
Advanced
Browse
Search
21231
25
Loading, please wait...
Cancel
software engineer
Title posted on indeed.com -
Software Engineer (SysAdmin)
Posted on 
				May 13, 2026
by
Employer details
TimePlay
Save to favourites
Your favourites
To add a job posting to your favourites, you need a Job Bank account. Sign in or sign up now!
Sign in
Sign up for a Plus account
Actions
Email
Copy link
Job details
*About TimePlay*

If you?ve 

In [ ]:
[# according to our company who is a .....
    "Software Engineer", "Software Developer", "Full Stack Developer",
    "Backend Developer", "Frontend Developer", "Data Scientist",
    "Machine Learning Engineer", "AI Engineer", "DevOps Engineer",
    "Cloud Engineer", "Data Analyst", "Business Intelligence Analyst",
    "QA Engineer", "Test Automation Engineer", "iOS Developer",
    "Android Developer", "UI Designer", "UX Designer", "Product Designer",
    "Cybersecurity Analyst", "Python Developer", "Data Engineer",
    "Network Engineer", "Cloud Architect", "Systems Engineer",
    "Java Developer", ".NET Developer", "Web Developer", "SDET",
    "Solutions Architect", "Big Data Specialist", "Fintech Engineer",
    "AI Prompt Engineer", "Blockchain Developer", "Robotics Engineer",
    "Javascript Developer", "AR Developer", "VR Developer",
    "IoT Engineer", "Ethical Hacker", "SRE", "Game Developer",

    "Product Manager", "Project Manager", "Marketing Specialist",
    "Digital Marketing Specialist", "SEO Specialist", "Content Writer",
    "Copywriter", "Business Analyst", "Operations Manager",
    "Sales Executive", "Technical Writer", "Market Research Analyst",
    "Graphic Designer"
]




In [ ]:
"JobID"	Unique identifier for each job description
"Title"	Job role/title
"ExperienceLevel"	Fresher / Junior / Experienced / Lead / Senior
"YearsOfExperience"	Numeric range or years (e.g., 0-1, 3-5)
"Skills"	Semicolon-separated list of required skills
"Responsibilities"	Semicolon-separated list of key responsibilities
"Keywords"	Semicolon-separated list of role-specific focus areas
"SalaryRange"	Numeric range or specific salary (e.g., 50k-70k, 60k)
"Location"	City or region of the job


In [ ]:

def register_eval_dataset_to_ui(eval_rows, dataset_name=dataset_ui_name):

    exp = mlflow.get_experiment_by_name(exp_name)
    exp_id = exp.experiment_id

    try:
        dataset = create_dataset(name=dataset_name, experiment_id=[exp_id])
    except Exception:
        dataset = get_dataset(name=dataset_name)

    records = []
    for row in eval_rows:
        gt_dict = json.loads(row["ground_truth"]) 
        
        records.append({
            "inputs": {
                "article_text": row["article_text"],
                "filename": row["filename"]
            },
            "expectations": {
                "ground_truth_data": gt_dict,
                "guidelines": ["Extract year, author, and study count accurately."]
            }
        })

    dataset.merge_records(records)
    print(f"Dataset '{dataset_name}' successfully registered to the Registry.")

In [ ]:
model = OpenAIChatModel(
    model_id,
    provider=OpenAIProvider(
        base_url=os.getenv("OPENAI_API_BASE"),
        api_key=os.getenv("OPENAI_API_KEY"), 
    ),
    profile=ModelProfile(
        default_structured_output_mode='json',
        supports_json_schema_output=True,
    ),
)

extra_body_dict = json.loads(os.getenv("LITELLM_EXTRA_BODY", "{}"))
settings = ModelSettings(
    extra_body=extra_body_dict
)


extraction_agent = Agent(
    model,
    model_settings=settings,
    #system_prompt=new_template,
    #deps_type=ArticleReviewInput,
    #output_type=ArticleReviewOutput,
    retries=3 
)


In [12]:
model = OpenAIChatModel(
    model_id,
    provider=OpenAIProvider(
        base_url= os.getenv("OPENAI_API_BASE"),
        api_key=os.getenv("OPENAI_API_KEY"), 
    ),
    profile=ModelProfile(
        default_structured_output_mode='json',
        supports_json_schema_output=True,  # tells PydanticAI to send the schema
    ),
)
extra_body_dict = json.loads(os.getenv("LITELLM_EXTRA_BODY"))
settings =  ModelSettings(
    extra_body=extra_body_dict
)

In [32]:

# Path to your MCP server script
script_path = os.path.abspath("mcp_server.py")

# Create a toolset that connects to your running server
toolset = MCPToolset(script_path)


extraction_agent = Agent(
    model,
    model_settings=settings,
    #system_prompt=new_template,
    #deps_type=ArticleReviewInput,
    #output_type=ArticleReviewOutput,
    toolsets=[toolset],
    retries=3 
)


In [28]:
conversation_history = []

In [29]:
result1 = await extraction_agent.run(
    "Please look up the job description for systems_engineer.",
    #allow_tools=True
)

conversation_history = result1.all_messages()
print(result1.output)


The **Systems Engineer** in this automotive context serves as the logical architect of the vehicle, bridging the gap between mechanical design, electrical hardware, and software development. Their primary mission is to manage the complexity of vehicle features (e.g., adaptive cruise control, steering) by decomposing them into functional architectures across various Electronic Control Units (ECUs).

### Key Responsibilities & Workflow
*   **Requirement Management:** Authoring system requirements specifications and designing Interface Control Documents (ICDs) to define signal flow between ECUs.
*   **Architecture & Design:** Modeling functional behavior of subsystems and executing the standardized systems development lifecycle (**V-Model**).
*   **Safety & Compliance:** Collaborating with safety engineers on Hazard Analysis and Risk Assessments (HARA) and drafting functional safety concepts.
*   **Verification:** Working with test engineers to ensure physical system integration matches s

In [30]:
conversation_history

[ModelRequest(parts=[UserPromptPart(content='Please look up the job description for systems_engineer.', timestamp=datetime.datetime(2026, 5, 30, 16, 39, 4, 109396, tzinfo=datetime.timezone.utc))], timestamp=datetime.datetime(2026, 5, 30, 16, 39, 4, 109396, tzinfo=datetime.timezone.utc), run_id='019e79c0-e0a5-76c6-afc0-03817d7a54de', conversation_id='019e79c0-e0a5-76c6-afc0-03805a301d42'),
 ModelResponse(parts=[ToolCallPart(tool_name='get_job_description', args='{"job_key":"systems_engineer"}', tool_call_id='ikawnVlcaXyR0U2biYxu5oigfeXemLuo')], usage=RequestUsage(input_tokens=279, output_tokens=21), model_name='ggml-org/gemma-4-31B-it-GGUF:Q4_K_M', timestamp=datetime.datetime(2026, 5, 30, 16, 39, 5, 174872, tzinfo=datetime.timezone.utc), provider_name='openai', provider_url='https://apphubai.wolke.uni-greifswald.de/v1/', provider_details={'finish_reason': 'tool_calls', 'timestamp': datetime.datetime(2026, 5, 30, 16, 39, 3, tzinfo=TzInfo(0))}, provider_response_id='chatcmpl-0gZrwQkxpnane

In [ ]:
# --- TURN 2 ---
# Ask a follow-up question. The agent now has context of what it did in Turn 1
# because we pass 'conversation_history' back into 'message_history'.
result2 = await extraction_agent.run(
    "What are the non-negotiable toolchains for that role based on what you just looked up?",
    message_history=conversation_history
)

In [ ]:
# =====================================================================
# 4. Define MCP Toolsets (Subprocess / STDIO Transport)
# =====================================================================
# Runs 'server.py' in a managed background process communicating over stdio.
# Ensure the path pointing to server.py is accurate relative to this script.


# =====================================================================
# 5. Initialize Pydantic AI Agent
# =====================================================================
extraction_agent = Agent(
    model,
    model_settings=settings,
    deps_type=ArticleReviewInput,
    output_type=ArticleReviewOutput,
    # Register the local PDF scraping function as a tool
    tools=[scrape_pdf_content],
    # Register the external MCP server's tools
    toolsets=[mcp_toolset],
    retries=3 
)

In [ ]:
from pydantic_ai.mcp import MCPServerStdio

# Path to your MCP server script
script_path = os.path.abspath("mcp_server.py")

# Create a robust, long-lived client connection using 10-minute limits
mcp_server = MCPServerStdio(
    'python', 
    [script_path],
    timeout=600.0,       # Max time in seconds to wait for initial handshake
    read_timeout=600.0   # Max time in seconds to wait for tool responses before dropping
)

extraction_agent = Agent(
    model,
    model_settings=settings,
    toolsets=[mcp_server],  # Use the configured mcp_server instead of the default toolset wrapper
    retries=3 
)

In [3]:
from orchestrator import run_orchestrator_chat


# The user's initial request
user_prompt = """ '
    I have a candidate CV located at 'cv/data/ENGINEERING/12011623.pdf'. Please parse this profile, search for 2 matching jobs remotely or in Germany, 
    evaluate them, and give me a ranked comparison table of the best fits.
"""

In [4]:
result = await run_orchestrator_chat(user_prompt = user_prompt)
result

UnsupportedOperation: fileno

In [ ]:
# Create the official OpenAI client for the sub-agents
mcp_provider = OpenAIProvider(
    base_url=os.getenv("OPENAI_API_BASE"),
    api_key=os.getenv("OPENAI_API_KEY"), 
    http_client=subagent_debug_client  # <--- CORRECT PLACEMENT
)

model = OpenAIChatModel(
    model_id,
    provider=mcp_provider, # <--- Pass the configured provider to the model
    profile=ModelProfile(
        default_structured_output_mode='tool',
        supports_json_schema_output=False, 
    ),
)